In [ ]:
import pandas as pd
import numpy as np
import os
import zipfile
import shutil
import chardet
from scipy.stats import norm
from google.colab import drive

# 1. 掛載 Google Drive
drive.mount('/content/drive')

# 設定路徑 (請根據你的雲端硬碟實際路徑修改)
base_path = '/content/drive/MyDrive/金融資料探勘/2022逐筆交易資料/Option_2022/'
temp_extract_path = '/content/temp_options/' # Colab 本地臨時解壓目錄

# 建立臨時目錄
if not os.path.exists(temp_extract_path):
    os.makedirs(temp_extract_path)

# 2. Black-Scholes 與 二分法函式
def bs_call_price(S, K, T, r, sigma):
    if T <= 0 or sigma <= 0: return 0
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

def calculate_iv(market_price, S, K, T, r):
    low, high = 0.0001, 5.0
    for _ in range(100):
        mid = (low + high) / 2
        if abs(bs_call_price(S, K, T, r, mid) - market_price) < 1e-5:
            return mid
        if bs_call_price(S, K, T, r, mid) < market_price:
            low = mid
        else:
            high = mid
    return mid

# 3. 讀取索引檔
index_df = pd.read_csv(os.path.join(base_path, 'Index_411336004_2022.csv'))

results = []

# 4. 迴圈處理每一天的檔案
for _, row in index_df.iterrows():
    # 建立目標資料夾路徑，例如：.../OptionsDaily_2022_01_05/
    folder_name = row['File'].replace('.csv', '')
    target_dir = os.path.join(base_path, folder_name)

    if not os.path.exists(target_dir):
        print(f"跳過：找不到目錄 {target_dir}")
        continue

    # 搜尋該目錄下的 zip 檔
    zip_files = [f for f in os.listdir(target_dir) if f.endswith('.zip')]

    csv_file_path = None

    if zip_files:
        # 執行解壓縮
        for zf in zip_files:
            with zipfile.ZipFile(os.path.join(target_dir, zf), 'r') as zip_ref:
                zip_ref.extractall(temp_extract_path)
                # 假設解壓後檔名與索引檔中記載的 File 欄位一致
                csv_file_path = os.path.join(temp_extract_path, row['File'])
    else:
        # 如果本來就是 CSV
        potential_csv = os.path.join(target_dir, row['File'])
        if os.path.exists(potential_csv):
            csv_file_path = potential_csv

    # 5. 讀取與計算
    if csv_file_path and os.path.exists(csv_file_path):
        try:
            # 自動偵測編碼
            with open(csv_file_path, 'rb') as f:
                enc = chardet.detect(f.read(10000))['encoding']

            df = pd.read_csv(csv_file_path, encoding=enc)

            # 篩選特定合約 (Contract)
            # 注意：請確認 daily CSV 內的欄位名稱，此處假設為 '到期月份(週別)'
            target_mask = (df['到期月份(週別)'].astype(str) == str(row['Contract'])) & (df['買賣權'] == 'Call')
            day_data = df[target_mask].copy()

            if not day_data.empty:
                S0, r, T = row['S0'], row['Rf']/100, row['Maturity']/252
                day_data['IV'] = day_data.apply(lambda x: calculate_iv(x['成交價格'], S0, x['履約價'], T, r), axis=1)
                day_data['Date'] = row['Date']
                results.append(day_data)
                print(f"成功處理：{row['Date']}")

            # 讀取完後刪除臨時 CSV 釋放空間
            if temp_extract_path in csv_file_path:
                os.remove(csv_file_path)

        except Exception as e:
            print(f"處理 {row['Date']} 時出錯: {e}")

# 6. 存檔
if results:
    final_output = pd.concat(results)
    final_output.to_csv('Final_IV_Analysis_2022.csv', index=False, encoding='utf-8-sig')
    print("分析完成！結果已儲存。")

# 清理臨時資料夾
shutil.rmtree(temp_extract_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/金融資料探勘/2022逐筆交易資料/Option_2022/Index_411336004_2022.csv'

In [ ]:
import os
# 請確認這個路徑與你雲端硬碟中的資料夾名稱「完全一模一樣」
check_path = '/content/drive/MyDrive/金融資料探勘/2022逐筆交易資料/Option_2022/'

if os.path.exists(check_path):
    print("目錄存在，檔案清單：")
    print(os.listdir(check_path))
else:
    print("找不到目錄，請檢查資料夾名稱（包含空格或全形字）")

找不到目錄，請檢查資料夾名稱（包含空格或全形字）


In [ ]:
import pandas as pd
import numpy as np
import os
from scipy.stats import norm
from google.colab import drive
from google.colab import files

# ==========================================
# 1. 雲端掛載與索引檔讀取
# ==========================================
drive.mount('/content/drive')

# 索引檔處理
index_file = 'Index_411336004_2022.csv'
if not os.path.exists(index_file):
    print("請上傳索引檔...")
    uploaded = files.upload()
    index_file = list(uploaded.keys())[0]

df_index = pd.read_csv(index_file)

# 基礎路徑 (依照您的描述)
drive_base_path = '/content/drive/MyDrive/金融資料探勘/2022逐筆交易資料/'

# ==========================================
# 2. Black-Scholes 與 二分法核心函數
# ==========================================
def bs_call_price(S, K, T, r, sigma):
    if T <= 0: return max(0.0, S - K)
    sigma = max(sigma, 1e-6)
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

def calculate_iv(market_price, S, K, T, r):
    # 內含價值檢查
    if market_price <= max(0, S - K * np.exp(-r * T)):
        return np.nan
    low, high = 0.0001, 5.0
    for i in range(100):
        mid = (low + high) / 2
        price = bs_call_price(S, K, T, r, mid)
        if abs(market_price - price) < 1e-6:
            return mid
        if price < market_price: low = mid
        else: high = mid
    return (low + high) / 2

# ==========================================
# 3. 批次自動化處理
# ==========================================
all_results = []

print("開始自動掃描並處理 2022 逐筆資料...")

for idx, row in df_index.iterrows():
    f_name = row['File']
    folder_name = f_name.replace('.csv', '')
    csv_path = os.path.join(drive_base_path, folder_name, f_name)

    if not os.path.exists(csv_path):
        continue

    # A. 讀取資料 (解決 DtypeWarning)
    try:
        df_tick = pd.read_csv(csv_path, encoding='cp950', low_memory=False)
    except:
        df_tick = pd.read_csv(csv_path, encoding='utf-8-sig', low_memory=False)

    # B. 強力清洗與自動欄位匹配 (解決 KeyError)
    df_tick.columns = [str(c).strip().replace('\ufeff', '') for c in df_tick.columns]

    def find_target_col(keywords):
        for col in df_tick.columns:
            if any(k in col for k in keywords): return col
        return None

    # 自動識別欄位名稱
    col_pc = find_target_col(['買賣權', '買賣別', 'CallPut', 'CP'])
    col_exp = find_target_col(['到期月份', '合約', 'Contract'])
    col_prc = find_target_col(['成交價格', '成交價', 'Price'])
    col_strk = find_target_col(['履約價', 'Strike'])

    # 檢查是否匹配成功
    if not all([col_pc, col_exp, col_prc, col_strk]):
        print(f"警告 [{f_name}]: 欄位無法自動匹配。現有欄位: {df_tick.columns.tolist()}")
        continue

    # C. 資料篩選 (轉換型態確保比對成功)
    target_contract = str(row['Contract']).strip()
    df_filtered = df_tick[
        (df_tick[col_pc].astype(str).str.contains('Call|C', case=False, na=False)) &
        (df_tick[col_exp].astype(str).str.strip() == target_contract)
    ].copy()

    # D. 計算 IV
    if not df_filtered.empty:
        S, rf, T = row['S0'], row['Rf']/100, row['Maturity']/252

        # 進行 IV 運算
        df_filtered['IV'] = df_filtered.apply(
            lambda x: calculate_iv(x[col_prc], S, x[col_strk], T, rf), axis=1
        )

        df_filtered['TradingDate'] = row['Date']
        # 僅保留核心結果以節省記憶體
        all_results.append(df_filtered[['TradingDate', col_exp, col_strk, col_prc, 'IV']])
        print(f"成功: {row['Date']} (處理 {len(df_filtered)} 筆)")

# ==========================================
# 4. 合併結果與匯出
# ==========================================
if all_results:
    final_df = pd.concat(all_results, ignore_index=True)
    final_df.to_csv('Final_IV_Analysis_2022.csv', index=False, encoding='utf-8-sig')
    print("\n--- 任務完成 ---")
    files.download('Final_IV_Analysis_2022.csv')
else:
    print("\n[錯誤] 未能成功處理任何資料，請檢查路徑與欄位名稱。")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
開始自動掃描並處理 2022 逐筆資料...
成功: 2022/1/3 (處理 32957 筆)
成功: 2022/1/4 (處理 47743 筆)
成功: 2022/1/5 (處理 35370 筆)


KeyboardInterrupt: 

In [ ]:
import pandas as pd
import numpy as np
import os
from scipy.stats import norm
from google.colab import drive
from google.colab import files

# ==========================================
# 1. 雲端掛載與索引檔讀取
# ==========================================
drive.mount('/content/drive')

index_file = 'Index_411336004_2022.csv'
if not os.path.exists(index_file):
    print("請上傳索引檔...")
    uploaded = files.upload()
    index_file = list(uploaded.keys())[0]

df_index = pd.read_csv(index_file)
drive_base_path = '/content/drive/MyDrive/金融資料探勘/2022逐筆交易資料/'

# ==========================================
# 2. Black-Scholes 與 IV 計算核心
# ==========================================
def bs_call_price(S, K, T, r, sigma):
    if T <= 0: return max(0.0, S - K)
    sigma = max(sigma, 1e-6)
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

def calculate_iv(market_price, S, K, T, r):
    if market_price <= max(0, S - K * np.exp(-r * T)):
        return np.nan
    low, high = 0.0001, 5.0
    for i in range(100):
        mid = (low + high) / 2
        price = bs_call_price(S, K, T, r, mid)
        if abs(market_price - price) < 1e-6:
            return mid
        if price < market_price: low = mid
        else: high = mid
    return (low + high) / 2

# ==========================================
# 3. 批次處理與過濾
# ==========================================
all_results = []

print("開始處理 2022 逐筆資料 (過濾交易量 > 30)...")

for idx, row in df_index.iterrows():
    f_name = row['File']
    folder_name = f_name.replace('.csv', '')
    csv_path = os.path.join(drive_base_path, folder_name, f_name)

    if not os.path.exists(csv_path):
        continue

    # A. 讀取並清洗欄位
    try:
        df_tick = pd.read_csv(csv_path, encoding='cp950', low_memory=False)
    except:
        df_tick = pd.read_csv(csv_path, encoding='utf-8-sig', low_memory=False)

    # 清除欄位空格與 BOM 字元
    df_tick.columns = [str(c).strip().replace('\ufeff', '') for c in df_tick.columns]

    # B. 定義欄位查找函數
    def find_col(keywords):
        for col in df_tick.columns:
            if any(k in col for k in keywords): return col
        return None

    # 自動匹配欄位
    col_pc = find_col(['買賣權', '買賣別', 'CallPut'])
    col_exp = find_col(['到期月份', '合約'])
    col_prc = find_col(['成交價格', '成交價'])
    col_strk = find_col(['履約價'])
    col_vol = find_col(['成交數量', '成交量', 'Qty', 'Volume'])

    # 檢查必要欄位是否齊全
    if not all([col_pc, col_exp, col_prc, col_strk, col_vol]):
        print(f"跳過 {f_name}: 欄位不全。現有: {list(df_tick.columns)}")
        continue

    # C. 篩選 (Call + 正確合約 + 交易量 > 30)
    target_contract = str(row['Contract']).strip()

    # 清洗成交數量欄位並轉數值
    df_tick[col_vol] = pd.to_numeric(df_tick[col_vol].astype(str).str.strip(), errors='coerce')

    df_filtered = df_tick[
        (df_tick[col_pc].astype(str).str.contains('Call|C', case=False, na=False)) &
        (df_tick[col_exp].astype(str).str.strip() == target_contract) &
        (df_tick[col_vol] > 30)
    ].copy()

    # D. 計算 IV
    if not df_filtered.empty:
        S, rf, T = row['S0'], row['Rf']/100, row['Maturity']/252
        df_filtered['IV'] = df_filtered.apply(
            lambda x: calculate_iv(x[col_prc], S, x[col_strk], T, rf), axis=1
        )
        df_filtered['TradingDate'] = row['Date']

        # 保留結果欄位
        all_results.append(df_filtered[['TradingDate', col_exp, col_strk, col_prc, col_vol, 'IV']])
        print(f"完成: {row['Date']} (處理筆數: {len(df_filtered)})")

# ==========================================
# 4. 合併結果與敘述性統計
# ==========================================
if all_results:
    final_df = pd.concat(all_results, ignore_index=True)

    # 針對每日 IV 進行敘述性統計
    daily_stats = final_df.groupby('TradingDate')['IV'].describe()

    # 匯出檔案
    final_df.to_csv('Option_IV_Results_Filtered.csv', index=False, encoding='utf-8-sig')
    daily_stats.to_csv('Daily_IV_Statistics_Summary.csv', encoding='utf-8-sig')

    print("\n" + "="*30)
    print("每日 IV 統計摘要 (前五日):")
    print(daily_stats.head())

    # 下載檔案
    files.download('Option_IV_Results_Filtered.csv')
    files.download('Daily_IV_Statistics_Summary.csv')
    print("\n處理完成，報告已自動下載。")
else:
    print("\n[錯誤] 找不到符合條件 (交易量 > 30) 的資料。")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
開始處理 2022 逐筆資料 (過濾交易量 > 30)...
完成: 2022/1/3 (處理筆數: 208)
完成: 2022/1/4 (處理筆數: 427)
完成: 2022/1/5 (處理筆數: 188)
完成: 2022/1/6 (處理筆數: 262)
完成: 2022/1/7 (處理筆數: 269)
完成: 2022/1/10 (處理筆數: 176)
完成: 2022/1/11 (處理筆數: 252)
完成: 2022/1/12 (處理筆數: 583)
完成: 2022/1/19 (處理筆數: 111)
完成: 2022/1/20 (處理筆數: 244)
完成: 2022/1/21 (處理筆數: 194)
完成: 2022/1/24 (處理筆數: 156)
完成: 2022/1/25 (處理筆數: 170)
完成: 2022/1/26 (處理筆數: 289)
完成: 2022/2/7 (處理筆數: 521)
完成: 2022/2/8 (處理筆數: 327)
完成: 2022/2/9 (處理筆數: 397)
完成: 2022/2/16 (處理筆數: 103)
完成: 2022/2/17 (處理筆數: 235)
完成: 2022/2/18 (處理筆數: 79)
完成: 2022/2/21 (處理筆數: 167)
完成: 2022/2/22 (處理筆數: 158)
完成: 2022/2/23 (處理筆數: 122)
完成: 2022/2/24 (處理筆數: 204)
完成: 2022/2/25 (處理筆數: 148)
完成: 2022/3/1 (處理筆數: 194)
完成: 2022/3/2 (處理筆數: 77)
完成: 2022/3/3 (處理筆數: 192)
完成: 2022/3/4 (處理筆數: 207)
完成: 2022/3/7 (處理筆數: 457)
完成: 2022/3/8 (處理筆數: 235)
完成: 2022/3/9 (處理筆數: 341)
完成: 2022/3/16 (處理筆數: 184)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


處理完成，報告已自動下載。


In [ ]:
import pandas as pd
import numpy as np
import os
import zipfile
import shutil
import chardet
from scipy.stats import norm
from google.colab import drive

# 1. 掛載 Google Drive
drive.mount('/content/drive')

# --- 路徑設定 (請確認與你的雲端硬碟路徑一致) ---
base_path = '/content/drive/MyDrive/金融資料探勘/2022逐筆交易資料/Option_2022/'
index_name = 'Index_411336004_2022.csv'
temp_extract_path = '/content/temp_options/' # Colab 本地臨時目錄

# 2. Black-Scholes 與 IV 計算函式
def bs_call_price(S, K, T, r, sigma):
    if T <= 0 or sigma <= 0: return 0
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

def calculate_iv(market_price, S, K, T, r):
    low, high = 0.0001, 5.0 # 0.01% 到 500%
    for _ in range(100):
        mid = (low + high) / 2
        price = bs_call_price(S, K, T, r, mid)
        if abs(price - market_price) < 1e-5:
            return mid
        if price < market_price:
            low = mid
        else:
            high = mid
    return mid

# 3. 讀取索引檔
index_path = os.path.join(base_path, index_name)
if not os.path.exists(index_path):
    raise FileNotFoundError(f"找不到索引檔：{index_path}")

index_df = pd.read_csv(index_path)
all_days_data = []

# 建立臨時解壓資料夾
if os.path.exists(temp_extract_path): shutil.rmtree(temp_extract_path)
os.makedirs(temp_extract_path)

print("開始處理資料...")

# 4. 逐日處理
for _, row in index_df.iterrows():
    trade_date = row['Date']
    target_contract = str(row['Contract'])
    S0, r_val, T_days = row['S0'], row['Rf']/100, row['Maturity']
    T = T_days / 252 # 年化到期時間

    # 定位資料夾 (例如: OptionsDaily_2022_01_03)
    folder_name = row['File'].replace('.csv', '')
    daily_folder = os.path.join(base_path, folder_name)

    if not os.path.exists(daily_folder):
        continue

    # 尋找並解壓縮 zip 檔
    csv_file = None
    zips = [f for f in os.listdir(daily_folder) if f.endswith('.zip')]

    try:
        if zips:
            for z in zips:
                with zipfile.ZipFile(os.path.join(daily_folder, z), 'r') as zip_ref:
                    zip_ref.extractall(temp_extract_path)
            # 解壓後找到對應的 CSV 檔案
            csv_file = os.path.join(temp_extract_path, row['File'])
        else:
            # 如果沒有 zip，直接找 CSV
            potential_csv = os.path.join(daily_folder, row['File'])
            if os.path.exists(potential_csv):
                csv_file = potential_csv

        if csv_file and os.path.exists(csv_file):
            # 自動偵測編碼
            with open(csv_file, 'rb') as f:
                enc = chardet.detect(f.read(10000))['encoding']

            # 讀取逐筆交易資料
            df = pd.read_csv(csv_file, encoding=enc)

            # 清理欄位名稱 (移除空格)
            df.columns = df.columns.str.strip()

            # 5. 資料篩選與欄位轉換
            # 篩選條件：Call、正確的到期月份
            mask = (df['買賣權'] == 'Call') & (df['到期月份(週別)'].astype(str) == target_contract)
            day_subset = df[mask].copy()

            if not day_subset.empty:
                # 計算該日每筆交易的 IV
                # 注意：這裡假設原始欄位為 '履約價', '成交價格', '成交量'
                day_subset['IV'] = day_subset.apply(
                    lambda x: calculate_iv(x['成交價格'], S0, x['履約價'], T, r_val), axis=1
                )

                # 計算該日平均 IV
                daily_avg_iv = day_subset['IV'].mean()

                # 重新整理成你需要的欄位名稱
                day_subset['Date'] = trade_date
                day_subset['Strike'] = day_subset['履約價']
                day_subset['Price'] = day_subset['成交價格']
                day_subset['Vol'] = day_subset['成交量']
                day_subset['日平均IV'] = daily_avg_iv

                # 只保留你要求的欄位
                output_subset = day_subset[['Date', 'Strike', 'Price', 'Vol', 'IV', '日平均IV']]
                all_days_data.append(output_subset)
                print(f"成功處理 {trade_date}，平均 IV: {daily_avg_iv:.4f}")

            # 處理完當天後，刪除臨時檔案釋放空間
            for f in os.listdir(temp_extract_path):
                os.remove(os.path.join(temp_extract_path, f))

    except Exception as e:
        print(f"處理 {trade_date} 時發生錯誤: {e}")

# 6. 合併與輸出
if all_days_data:
    final_df = pd.concat(all_days_data, ignore_index=True)
    output_filename = 'Options_Analysis_Result_2022.csv'
    final_df.to_csv(output_filename, index=False, encoding='utf-8-sig')
    print(f"\n任務完成！結果已儲存至: {output_filename}")
else:
    print("\n未找到符合條件的資料，請檢查路徑與欄位名稱。")

# 清理暫存資料夾
shutil.rmtree(temp_extract_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


FileNotFoundError: 找不到索引檔：/content/drive/MyDrive/金融資料探勘/2022逐筆交易資料/Option_2022/Index_411336004_2022.csv

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

# 1. 檢查基礎目錄是否存在
search_path = '/content/drive/MyDrive/'
print("--- 正在掃描您的雲端硬碟根目錄 ---")
print(os.listdir(search_path))

# 2. 自動尋找檔案功能 (如果不確定在哪，這段會幫你搜)
def find_file(name, path):
    for root, dirs, files in os.walk(path):
        if name in files:
            return os.path.join(root, name)
    return None

target_file = 'Index_411336004_2022.csv'
actual_path = find_file(target_file, '/content/drive/MyDrive/')

if actual_path:
    print(f"\n✅ 找到檔案了！請將 base_path 修改為：")
    print(f"base_path = '{os.path.dirname(actual_path)}/'")
else:
    print(f"\n❌ 找不到名為 {target_file} 的檔案，請確認檔案是否已上傳至雲端硬碟。")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
--- 正在掃描您的雲端硬碟根目錄 ---
['音樂', '照片', 'Screenshot_2018-09-02-18-41-37_1.jpg', 'IMAG0545.jpg', 'FB_IMG_1608547670703.jpg', 'FB_IMG_1608547662822.jpg', '20210503_225125.jpg', 'IMG_20201004_094022.jpg', 'Screenshot_2020-09-27-23-29-54-258_com.instagram.android.jpg', 'IMG_20210706_104159.jpg', '90422會考成績單掃描.pdf', '110學年度臺北市立陽明高級中學雙語實驗班甄選書審資料-劉芳孜.docx', 'IMG_20210830_184759174.jpg', '9zVOPLF.jpg', '螢幕擷取畫面 (11).png', '螢幕擷取畫面 (12).png', 'GREEN SCREEN MAIS USADOS NO MOMENTO PELOS YOUTUBERS❤.mp4', 'LINE_MOVIE_1523785573292.mp4', 'HUGE PREMADE ENDSLATE PACK + giveaway winners.mp4', 'GREEN SCREEN MAIS USADOS PELOS YOUTUBERS_MELHORES GREEN SCREEN PARA A SUA EDIÇÃO.mp4', 'Animated Subscribe Button Green Screen _ EditingBabeX.mp4', '72040.jpg', '我', 'IMG_9461.PNG', 'IMG_9462.PNG', 'AI', 'Webtoon', '日本四國之旅.gmap', '波捏豆🚪', '未命名文件 (1).gdoc', 'Sport.gdoc', '未命名文件.gdoc', '課表.gdoc

In [ ]:
import pandas as pd
import numpy as np
import os
import zipfile
import shutil
import chardet
from scipy.stats import norm
from google.colab import drive

# 1. 掛載 Google Drive
drive.mount('/content/drive')

# --- 請根據步驟 1 找到的路徑進行修改 ---
base_path = '/content/drive/MyDrive/金融資料探勘/2022逐筆交易資料/Option_2022/'
index_name = 'Index_411336004_2022.csv'
temp_extract_path = '/content/temp_options/'

# 2. BS 模型與 IV 計算
def bs_call_price(S, K, T, r, sigma):
    if T <= 0 or sigma <= 0: return 0
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

def calculate_iv(market_price, S, K, T, r):
    low, high = 0.0001, 5.0
    for _ in range(50): # 迭代 50 次精度已足夠
        mid = (low + high) / 2
        if bs_call_price(S, K, T, r, mid) < market_price:
            low = mid
        else:
            high = mid
    return mid

# 3. 讀取索引檔
index_path = os.path.join(base_path, index_name)
if not os.path.exists(index_path):
    # 如果路徑還是不對，嘗試手動從 Colab 左側檔案欄複製路徑貼過來
    raise FileNotFoundError(f"路徑錯誤，請確認檔案位置：{index_path}")

index_df = pd.read_csv(index_path)
all_days_data = []

if os.path.exists(temp_extract_path): shutil.rmtree(temp_extract_path)
os.makedirs(temp_extract_path)

# 4. 處理資料
for _, row in index_df.iterrows():
    trade_date = row['Date']
    target_contract = str(row['Contract'])
    S0, r_val, T = row['S0'], row['Rf']/100, row['Maturity']/252

    folder_name = row['File'].replace('.csv', '')
    daily_folder = os.path.join(base_path, folder_name)

    if not os.path.exists(daily_folder): continue

    zips = [f for f in os.listdir(daily_folder) if f.endswith('.zip')]
    csv_file = None

    try:
        if zips:
            for z in zips:
                with zipfile.ZipFile(os.path.join(daily_folder, z), 'r') as zip_ref:
                    zip_ref.extractall(temp_extract_path)
            csv_file = os.path.join(temp_extract_path, row['File'])
        else:
            potential_csv = os.path.join(daily_folder, row['File'])
            if os.path.exists(potential_csv): csv_file = potential_csv

        if csv_file and os.path.exists(csv_file):
            with open(csv_file, 'rb') as f:
                enc = chardet.detect(f.read(10000))['encoding']

            df = pd.read_csv(csv_file, encoding=enc)
            df.columns = df.columns.str.strip()

            # 篩選 Call 與 對應月份
            mask = (df['買賣權'] == 'Call') & (df['到期月份(週別)'].astype(str) == target_contract)
            day_subset = df[mask].copy()

            if not day_subset.empty:
                # 計算 IV
                day_subset['IV'] = day_subset.apply(
                    lambda x: calculate_iv(x['成交價格'], S0, x['履約價'], T, r_val), axis=1
                )

                # 計算日平均 IV
                daily_avg = day_subset['IV'].mean()

                # 重新命名與整理欄位
                day_subset['Date'] = trade_date
                day_subset['Strike'] = day_subset['履約價']
                day_subset['Price'] = day_subset['成交價格']
                day_subset['Vol'] = day_subset['成交量']
                day_subset['日平均IV'] = daily_avg

                # 選取要求的欄位
                output = day_subset[['Date', 'Strike', 'Price', 'Vol', 'IV', '日平均IV']]
                all_days_data.append(output)
                print(f"完成: {trade_date}")

            # 清理臨時檔
            for f in os.listdir(temp_extract_path): os.remove(os.path.join(temp_extract_path, f))
    except Exception as e:
        print(f"跳過 {trade_date}: {e}")

# 5. 輸出結果
if all_days_data:
    final_res = pd.concat(all_days_data)
    final_res.to_csv('Option_Analysis_Final.csv', index=False, encoding='utf-8-sig')
    print("分析成功！請在左側檔案選單下載 Option_Analysis_Final.csv")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


FileNotFoundError: 路徑錯誤，請確認檔案位置：/content/drive/MyDrive/金融資料探勘/2022逐筆交易資料/Option_2022/Index_411336004_2022.csv

In [ ]:
import pandas as pd
import numpy as np
import os
import zipfile
import shutil
import chardet
from scipy.stats import norm
from google.colab import drive

# 1. 掛載 Google Drive
drive.mount('/content/drive')

# 2. 自動尋找索引檔路徑 (避免手打路徑錯誤)
def find_target_file(filename, search_path='/content/drive/MyDrive/'):
    print(f"正在搜尋檔案 {filename}...")
    for root, dirs, files in os.walk(search_path):
        if filename in files:
            return os.path.join(root, filename)
    return None

index_filename = 'Index_411336004_2022.csv'
full_index_path = find_target_file(index_filename)

if full_index_path is None:
    raise FileNotFoundError(f"在雲端硬碟中完全找不到 {index_filename}，請確認檔案已上傳。")

# 設定基準路徑 (索引檔所在的資料夾)
base_path = os.path.dirname(full_index_path)
temp_extract_path = '/content/temp_options/'
print(f"✅ 找到索引檔於: {full_index_path}")
print(f"📂 基準路徑設定為: {base_path}")

# 3. Black-Scholes 與 IV 計算
def bs_call_price(S, K, T, r, sigma):
    if T <= 0 or sigma <= 0: return 0
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

def calculate_iv(market_price, S, K, T, r):
    low, high = 0.0001, 5.0
    for _ in range(50):
        mid = (low + high) / 2
        if bs_call_price(S, K, T, r, mid) < market_price:
            low = mid
        else:
            high = mid
    return mid

# 4. 讀取索引檔並處理
index_df = pd.read_csv(full_index_path)
all_days_data = []

if os.path.exists(temp_extract_path): shutil.rmtree(temp_extract_path)
os.makedirs(temp_extract_path)

for _, row in index_df.iterrows():
    trade_date = row['Date']
    target_contract = str(row['Contract'])
    S0, r_val, T = row['S0'], row['Rf']/100, row['Maturity']/252

    # 根據索引檔 File 欄位尋找對應資料夾
    folder_name = row['File'].replace('.csv', '')
    daily_folder = os.path.join(base_path, folder_name)

    if not os.path.exists(daily_folder):
        continue

    # 解壓縮邏輯
    zips = [f for f in os.listdir(daily_folder) if f.endswith('.zip')]
    csv_file = None

    try:
        if zips:
            for z in zips:
                with zipfile.ZipFile(os.path.join(daily_folder, z), 'r') as zip_ref:
                    zip_ref.extractall(temp_extract_path)
            csv_file = os.path.join(temp_extract_path, row['File'])
        else:
            potential_csv = os.path.join(daily_folder, row['File'])
            if os.path.exists(potential_csv): csv_file = potential_csv

        if csv_file and os.path.exists(csv_file):
            with open(csv_file, 'rb') as f:
                enc = chardet.detect(f.read(10000))['encoding']

            df = pd.read_csv(csv_file, encoding=enc)
            df.columns = df.columns.str.strip()

            mask = (df['買賣權'] == 'Call') & (df['到期月份(週別)'].astype(str) == target_contract)
            day_subset = df[mask].copy()

            if not day_subset.empty:
                day_subset['IV'] = day_subset.apply(
                    lambda x: calculate_iv(x['成交價格'], S0, x['履約價'], T, r_val), axis=1
                )
                daily_avg = day_subset['IV'].mean()

                # 設定你要求的欄位
                day_subset['Date'] = trade_date
                day_subset['Strike'] = day_subset['履約價']
                day_subset['Price'] = day_subset['成交價格']
                day_subset['Vol'] = day_subset['成交量']
                day_subset['日平均IV'] = daily_avg

                all_days_data.append(day_subset[['Date', 'Strike', 'Price', 'Vol', 'IV', '日平均IV']])
                print(f"已完成: {trade_date}")

            for f in os.listdir(temp_extract_path): os.remove(os.path.join(temp_extract_path, f))
    except Exception as e:
        print(f"錯誤 {trade_date}: {e}")

# 5. 輸出
if all_days_data:
    final_res = pd.concat(all_days_data)
    final_res.to_csv('Option_Analysis_Final.csv', index=False, encoding='utf-8-sig')
    print("\n✅ 分析成功！請在左側檔案選單下載 Option_Analysis_Final.csv")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
正在搜尋檔案 Index_411336004_2022.csv...
✅ 找到索引檔於: /content/drive/MyDrive/金融資料探勘/Index_411336004_2022.csv
📂 基準路徑設定為: /content/drive/MyDrive/金融資料探勘


In [ ]:
from google.colab import files
files.download('Option_Analysis_Final.csv')

FileNotFoundError: Cannot find file: Option_Analysis_Final.csv

In [ ]:
import pandas as pd
import numpy as np
import os
import zipfile
import shutil
import chardet
from scipy.stats import norm
from google.colab import drive, files

# 1. 掛載 Google Drive
drive.mount('/content/drive')

# 2. 尋找索引檔
def find_target_file(filename, search_path='/content/drive/MyDrive/'):
    print(f"🔍 正在搜尋索引檔 {filename}...")
    for root, dirs, files_in_dir in os.walk(search_path):
        if filename in files_in_dir:
            return os.path.join(root, filename)
    return None

index_filename = 'Index_411336004_2022.csv'
full_index_path = find_target_file(index_filename)

if full_index_path is None:
    print("❌ 錯誤：在雲端硬碟找不到 Index 檔案，請確認上傳路徑。")
else:
    base_path = os.path.dirname(full_index_path)
    print(f"✅ 找到索引檔：{full_index_path}")

    # 讀取索引檔
    index_df = pd.read_csv(full_index_path)
    print(f"📊 索引檔讀取成功，共 {len(index_df)} 筆交易日待處理。")

    # 3. 定義 BS 模型
    def bs_call_price(S, K, T, r, sigma):
        if T <= 0 or sigma <= 0: return 0
        d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
        d2 = d1 - sigma * np.sqrt(T)
        return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

    def calculate_iv(market_price, S, K, T, r):
        low, high = 0.0001, 5.0
        for _ in range(50):
            mid = (low + high) / 2
            if bs_call_price(S, K, T, r, mid) < market_price: low = mid
            else: high = mid
        return mid

    all_days_data = []
    temp_extract_path = '/content/temp_options/'
    if os.path.exists(temp_extract_path): shutil.rmtree(temp_extract_path)
    os.makedirs(temp_extract_path)

    # 4. 開始逐日處理
    for i, row in index_df.iterrows():
        trade_date = row['Date']
        target_contract = str(row['Contract'])
        S0, r_val, T = row['S0'], row['Rf']/100, row['Maturity']/252

        folder_name = row['File'].replace('.csv', '')
        daily_folder = os.path.join(base_path, folder_name)

        # 檢查資料夾是否存在
        if not os.path.exists(daily_folder):
            print(f"⚠️ 跳過 {trade_date}：找不到資料夾 {folder_name}")
            continue

        # 找 Zip 或 CSV
        zips = [f for f in os.listdir(daily_folder) if f.endswith('.zip')]
        csv_file = None

        try:
            if zips:
                for z in zips:
                    with zipfile.ZipFile(os.path.join(daily_folder, z), 'r') as zip_ref:
                        zip_ref.extractall(temp_extract_path)
                csv_file = os.path.join(temp_extract_path, row['File'])
            else:
                csv_file = os.path.join(daily_folder, row['File'])

            if csv_file and os.path.exists(csv_file):
                with open(csv_file, 'rb') as f:
                    enc = chardet.detect(f.read(10000))['encoding']

                df = pd.read_csv(csv_file, encoding=enc)
                df.columns = df.columns.str.strip()

                # --- 重要：檢查欄位名稱 ---
                if i == 0:
                    print(f"ℹ️ 第一份檔案欄位名稱為：{df.columns.tolist()}")

                # 篩選 (請確認欄位名稱是否與輸出的一致)
                mask = (df['買賣權'] == 'Call') & (df['到期月份(週別)'].astype(str) == target_contract)
                day_subset = df[mask].copy()

                if not day_subset.empty:
                    day_subset['IV'] = day_subset.apply(
                        lambda x: calculate_iv(x['成交價格'], S0, x['履約價'], T, r_val), axis=1
                    )
                    daily_avg = day_subset['IV'].mean()

                    # 整理欄位
                    day_subset['Date'] = trade_date
                    day_subset['Strike'] = day_subset['履約價']
                    day_subset['Price'] = day_subset['成交價格']
                    day_subset['Vol'] = day_subset['成交量']
                    day_subset['日平均IV'] = daily_avg

                    all_days_data.append(day_subset[['Date', 'Strike', 'Price', 'Vol', 'IV', '日平均IV']])
                    print(f"✅ {trade_date} 處理完畢，找到 {len(day_subset)} 筆資料")
                else:
                    print(f"❓ {trade_date} 篩選後沒有符合條件的 Call 資料")

            # 清理暫存
            if os.path.exists(temp_extract_path):
                for f in os.listdir(temp_extract_path): os.remove(os.path.join(temp_extract_path, f))

        except Exception as e:
            print(f"❌ {trade_date} 發生錯誤: {e}")

    # 5. 存檔與下載
    if all_days_data:
        final_res = pd.concat(all_days_data)
        output_name = 'Option_Analysis_Final.csv'
        final_res.to_csv(output_name, index=False, encoding='utf-8-sig')
        print(f"\n🎉 全部完成！檔案已產生：{output_name}")
        files.download(output_name)
    else:
        print("\n💀 失敗：最終資料清單是空的，沒有產生任何結果。")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🔍 正在搜尋索引檔 Index_411336004_2022.csv...
✅ 找到索引檔：/content/drive/MyDrive/金融資料探勘/Index_411336004_2022.csv
📊 索引檔讀取成功，共 246 筆交易日待處理。
⚠️ 跳過 2022/1/3：找不到資料夾 OptionsDaily_2022_01_03
⚠️ 跳過 2022/1/4：找不到資料夾 OptionsDaily_2022_01_04
⚠️ 跳過 2022/1/5：找不到資料夾 OptionsDaily_2022_01_05
⚠️ 跳過 2022/1/6：找不到資料夾 OptionsDaily_2022_01_06
⚠️ 跳過 2022/1/7：找不到資料夾 OptionsDaily_2022_01_07
⚠️ 跳過 2022/1/10：找不到資料夾 OptionsDaily_2022_01_10
⚠️ 跳過 2022/1/11：找不到資料夾 OptionsDaily_2022_01_11
⚠️ 跳過 2022/1/12：找不到資料夾 OptionsDaily_2022_01_12
⚠️ 跳過 2022/1/13：找不到資料夾 OptionsDaily_2022_01_13
⚠️ 跳過 2022/1/14：找不到資料夾 OptionsDaily_2022_01_14
⚠️ 跳過 2022/1/17：找不到資料夾 OptionsDaily_2022_01_17
⚠️ 跳過 2022/1/18：找不到資料夾 OptionsDaily_2022_01_18
⚠️ 跳過 2022/1/19：找不到資料夾 OptionsDaily_2022_01_19
⚠️ 跳過 2022/1/20：找不到資料夾 OptionsDaily_2022_01_20
⚠️ 跳過 2022/1/21：找不到資料夾 OptionsDaily_2022_01_21
⚠️ 跳過 2022/1/24：找不到資料夾 OptionsDaily_2022_01_2

In [ ]:
import pandas as pd
import numpy as np
import os
import zipfile
import shutil
import chardet
from scipy.stats import norm
from google.colab import drive, files

# 1. 掛載 Google Drive
drive.mount('/content/drive')

# 2. 自動定位 Index 檔案
def find_index_file(name='Index_411336004_2022.csv'):
    for root, dirs, files_in_dir in os.walk('/content/drive/MyDrive/'):
        if name in files_in_dir:
            return os.path.join(root, name)
    return None

full_index_path = find_index_file()

if not full_index_path:
    print("❌ 找不到索引檔，請確認檔案已上傳至雲端硬碟。")
else:
    base_path = os.path.dirname(full_index_path)
    print(f"✅ 找到索引檔：{full_index_path}")
    index_df = pd.read_csv(full_index_path)

    # 3. BS 模型與 IV 計算 (二分法)
    def bs_call_price(S, K, T, r, sigma):
        if T <= 0 or sigma <= 0: return 0
        d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
        d2 = d1 - sigma * np.sqrt(T)
        return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

    def calculate_iv(market_price, S, K, T, r):
        low, high = 0.0001, 5.0
        for _ in range(50):
            mid = (low + high) / 2
            if bs_call_price(S, K, T, r, mid) < market_price: low = mid
            else: high = mid
        return mid

    all_days_data = []
    temp_path = '/content/temp_extract/'
    if os.path.exists(temp_path): shutil.rmtree(temp_path)
    os.makedirs(temp_path)

    print("\n--- 開始處理逐日資料 ---")

    for i, row in index_df.iterrows():
        trade_date = row['Date']
        target_contract = str(row['Contract'])
        S0, r_val, T = row['S0'], row['Rf']/100, row['Maturity']/252

        # 根據你的說明：資料夾不帶 .csv
        folder_name = row['File'].replace('.csv', '')
        daily_folder = os.path.join(base_path, folder_name)

        if not os.path.exists(daily_folder):
            print(f"⚠️ 跳過 {trade_date}：找不到資料夾 {daily_folder}")
            continue

        # 找尋資料夾內的檔案 (優先解壓 zip)
        target_csv = None
        files_in_folder = os.listdir(daily_folder)

        try:
            zips = [f for f in files_in_folder if f.endswith('.zip')]
            if zips:
                for z in zips:
                    with zipfile.ZipFile(os.path.join(daily_folder, z), 'r') as zf:
                        zf.extractall(temp_path)
                # 解壓後，尋找 csv
                extracted_files = os.listdir(temp_path)
                target_csv = os.path.join(temp_path, row['File']) if row['File'] in extracted_files else None
            else:
                # 直接讀取資料夾內的 csv
                if row['File'] in files_in_folder:
                    target_csv = os.path.join(daily_folder, row['File'])

            if target_csv and os.path.exists(target_csv):
                with open(target_csv, 'rb') as f:
                    enc = chardet.detect(f.read(10000))['encoding']

                df = pd.read_csv(target_csv, encoding=enc)
                df.columns = df.columns.str.strip() # 去除欄位名稱空格

                # 自動辨識欄位 (避免中文名稱不合)
                col_type = next((c for c in df.columns if '買賣權' in c), None)
                col_exp = next((c for c in df.columns if '到期' in c), None)
                col_strike = next((c for c in df.columns if '履約' in c), None)
                col_price = next((c for c in df.columns if '成交價格' in c or '成交價' in c), None)
                col_vol = next((c for c in df.columns if '成交量' in c), None)

                if not all([col_type, col_exp, col_strike, col_price]):
                    print(f"❌ {trade_date} 欄位不完整: {df.columns.tolist()}")
                    continue

                # 篩選 Call 與 合約月份
                mask = (df[col_type].str.contains('Call', na=False)) & (df[col_exp].astype(str) == target_contract)
                subset = df[mask].copy()

                if not subset.empty:
                    # 計算 IV
                    subset['IV'] = subset.apply(
                        lambda x: calculate_iv(x[col_price], S0, x[col_strike], T, r_val), axis=1
                    )
                    daily_avg = subset['IV'].mean()

                    # 格式化輸出欄位
                    res = pd.DataFrame({
                        'Date': [trade_date] * len(subset),
                        'Strike': subset[col_strike],
                        'Price': subset[col_price],
                        'Vol': subset[col_vol] if col_vol else 0,
                        'IV': subset['IV'],
                        '日平均IV': daily_avg
                    })
                    all_days_data.append(res)
                    print(f"✅ {trade_date} 成功: 處理 {len(subset)} 筆")
                else:
                    print(f"❓ {trade_date} 篩選後無資料 (合約:{target_contract})")

            # 清理暫存
            for f in os.listdir(temp_path): os.remove(os.path.join(temp_path, f))

        except Exception as e:
            print(f"❌ {trade_date} 出錯: {e}")

    # 4. 存檔與下載
    if all_days_data:
        final_df = pd.concat(all_days_data, ignore_index=True)
        output_file = 'Option_Analysis_Final.csv'
        final_df.to_csv(output_file, index=False, encoding='utf-8-sig')
        print(f"\n🎉 完成！共處理 {len(all_days_data)} 天資料。")
        files.download(output_file)
    else:
        print("\n💀 失敗：沒有產生任何結果，請檢查上述錯誤訊息。")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 找到索引檔：/content/drive/MyDrive/金融資料探勘/Index_411336004_2022.csv

--- 開始處理逐日資料 ---
⚠️ 跳過 2022/1/3：找不到資料夾 /content/drive/MyDrive/金融資料探勘/OptionsDaily_2022_01_03
⚠️ 跳過 2022/1/4：找不到資料夾 /content/drive/MyDrive/金融資料探勘/OptionsDaily_2022_01_04
⚠️ 跳過 2022/1/5：找不到資料夾 /content/drive/MyDrive/金融資料探勘/OptionsDaily_2022_01_05
⚠️ 跳過 2022/1/6：找不到資料夾 /content/drive/MyDrive/金融資料探勘/OptionsDaily_2022_01_06
⚠️ 跳過 2022/1/7：找不到資料夾 /content/drive/MyDrive/金融資料探勘/OptionsDaily_2022_01_07
⚠️ 跳過 2022/1/10：找不到資料夾 /content/drive/MyDrive/金融資料探勘/OptionsDaily_2022_01_10
⚠️ 跳過 2022/1/11：找不到資料夾 /content/drive/MyDrive/金融資料探勘/OptionsDaily_2022_01_11
⚠️ 跳過 2022/1/12：找不到資料夾 /content/drive/MyDrive/金融資料探勘/OptionsDaily_2022_01_12
⚠️ 跳過 2022/1/13：找不到資料夾 /content/drive/MyDrive/金融資料探勘/OptionsDaily_2022_01_13
⚠️ 跳過 2022/1/14：找不到資料夾 /content/drive/MyDrive/金融資料探勘/OptionsDaily_2022_01_14
⚠️ 跳過 2022/1/17：找不到資料夾 /c

In [ ]:
import pandas as pd
import numpy as np
import os
import zipfile
import shutil
import chardet
from scipy.stats import norm
from google.colab import drive, files

# 1. 掛載 Google Drive
drive.mount('/content/drive')

# --- 手動指定路徑，確保不再跳過 ---
# 請從 Colab 左側資料夾找到 "Option_2022" 資料夾，右鍵「複製路徑」貼在下方
base_path = '/content/drive/MyDrive/金融資料探勘/2022逐筆交易資料/Option_2022/'
index_name = 'Index_411336004_2022.csv'
# ------------------------------

full_index_path = os.path.join(base_path, index_name)

if not os.path.exists(full_index_path):
    print(f"❌ 錯誤：在 {full_index_path} 找不到索引檔。")
    print("請檢查 base_path 是否正確。")
else:
    print(f"✅ 找到索引檔，準備處理資料...")
    index_df = pd.read_csv(full_index_path)

    # BS 與 IV 計算函式
    def bs_call_price(S, K, T, r, sigma):
        if T <= 0 or sigma <= 0: return 0
        d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
        d2 = d1 - sigma * np.sqrt(T)
        return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

    def calculate_iv(market_price, S, K, T, r):
        low, high = 0.0001, 5.0
        for _ in range(50):
            mid = (low + high) / 2
            if bs_call_price(S, K, T, r, mid) < market_price: low = mid
            else: high = mid
        return mid

    all_days_data = []
    temp_path = '/content/temp_extract/'
    if os.path.exists(temp_path): shutil.rmtree(temp_path)
    os.makedirs(temp_path)

    for i, row in index_df.iterrows():
        trade_date = row['Date']
        target_contract = str(row['Contract'])
        S0, r_val, T = row['S0'], row['Rf']/100, row['Maturity']/252

        # 修正：資料夾不帶 .csv
        folder_name = row['File'].replace('.csv', '')
        daily_folder = os.path.join(base_path, folder_name)

        if not os.path.exists(daily_folder):
            print(f"⚠️ 找不到資料夾，嘗試跳過 .csv 匹配: {daily_folder}")
            continue

        target_csv = None
        # 檢查資料夾內的所有檔案
        files_in_folder = os.listdir(daily_folder)

        try:
            # 優先處理 Zip 壓縮檔
            zips = [f for f in files_in_folder if f.endswith('.zip')]
            if zips:
                for z in zips:
                    with zipfile.ZipFile(os.path.join(daily_folder, z), 'r') as zf:
                        zf.extractall(temp_path)
                # 解壓後確認檔案是否存在
                if row['File'] in os.listdir(temp_path):
                    target_csv = os.path.join(temp_path, row['File'])

            # 若無 zip 或解壓失敗，直接找資料夾內的 CSV
            if not target_csv and row['File'] in files_in_folder:
                target_csv = os.path.join(daily_folder, row['File'])

            if target_csv:
                with open(target_csv, 'rb') as f:
                    enc = chardet.detect(f.read(10000))['encoding']

                df = pd.read_csv(target_csv, encoding=enc)
                df.columns = df.columns.str.strip()

                # 欄位自動匹配
                c_type = next((c for c in df.columns if '買賣權' in c), None)
                c_exp = next((c for c in df.columns if '到期' in c), None)
                c_strike = next((c for c in df.columns if '履約' in c), None)
                c_price = next((c for c in df.columns if '成交價格' in c or '成交價' in c), None)
                c_vol = next((c for c in df.columns if '成交量' in c), None)

                mask = (df[c_type].str.contains('Call', na=False)) & (df[c_exp].astype(str).str.contains(target_contract))
                subset = df[mask].copy()

                if not subset.empty:
                    subset['IV'] = subset.apply(lambda x: calculate_iv(x[c_price], S0, x[c_strike], T, r_val), axis=1)
                    daily_avg = subset['IV'].mean()

                    res = pd.DataFrame({
                        'Date': [trade_date] * len(subset),
                        'Strike': subset[c_strike],
                        'Price': subset[c_price],
                        'Vol': subset[c_vol] if c_vol else 0,
                        'IV': subset['IV'],
                        '日平均IV': daily_avg
                    })
                    all_days_data.append(res)
                    print(f"✅ {trade_date} 成功")

                # 清理暫存檔案
                if temp_path in target_csv:
                    for f in os.listdir(temp_path): os.remove(os.path.join(temp_path, f))

        except Exception as e:
            print(f"❌ {trade_date} 出錯: {e}")

    # 存檔與強制下載
    if all_days_data:
        final_df = pd.concat(all_days_data, ignore_index=True)
        out_name = 'Option_Analysis_Final.csv'
        # 同時存到 Colab 與 Google Drive
        final_df.to_csv(out_name, index=False, encoding='utf-8-sig')
        final_df.to_csv(os.path.join(base_path, out_name), index=False, encoding='utf-8-sig')
        print(f"\n🎉 完成！檔案已存至 Google Drive 資料夾下。")
        files.download(out_name)
    else:
        print("\n💀 仍未找到資料，請確認資料夾層級。")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
❌ 錯誤：在 /content/drive/MyDrive/金融資料探勘/2022逐筆交易資料/Option_2022/Index_411336004_2022.csv 找不到索引檔。
請檢查 base_path 是否正確。


In [ ]:
import pandas as pd
import numpy as np
import os
import zipfile
import shutil
import chardet
from scipy.stats import norm
from google.colab import drive, files

# 1. 掛載 Google Drive
drive.mount('/content/drive')

# 2. 自動尋找索引檔 (解決路徑錯誤的問題)
def find_path_auto(target_name):
    print(f"🔍 正在為您搜尋檔案：{target_name} ...")
    for root, dirs, files_in_dir in os.walk('/content/drive/MyDrive/'):
        if target_name in files_in_dir:
            return os.path.join(root, target_name)
    return None

index_filename = 'Index_411336004_2022.csv'
full_index_path = find_path_auto(index_filename)

if not full_index_path:
    print(f"❌ 找不到檔案 {index_filename}。請確認您已將檔案上傳到雲端硬碟。")
else:
    # 自動定義正確的 base_path
    base_path = os.path.dirname(full_index_path)
    print(f"✅ 成功定位！索引檔位於：{full_index_path}")
    print(f"📂 資料夾基準點已設為：{base_path}")

    # --- Black-Scholes 與 IV 計算核心 ---
    def bs_call_price(S, K, T, r, sigma):
        if T <= 0 or sigma <= 0: return 0
        d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
        d2 = d1 - sigma * np.sqrt(T)
        return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

    def calculate_iv(market_price, S, K, T, r):
        low, high = 0.0001, 5.0
        for _ in range(50):
            mid = (low + high) / 2
            if bs_call_price(S, K, T, r, mid) < market_price: low = mid
            else: high = mid
        return mid

    # 3. 讀取與處理
    index_df = pd.read_csv(full_index_path)
    all_days_data = []
    temp_path = '/content/temp_extract/'
    if os.path.exists(temp_path): shutil.rmtree(temp_path)
    os.makedirs(temp_path)

    for i, row in index_df.iterrows():
        trade_date = row['Date']
        target_contract = str(row['Contract'])
        S0, r_val, T = row['S0'], row['Rf']/100, row['Maturity']/252

        # 處理資料夾名稱 (移除 .csv)
        folder_name = row['File'].replace('.csv', '')
        daily_folder = os.path.join(base_path, folder_name)

        if not os.path.exists(daily_folder):
            # 備援方案：如果資料夾在 base_path 的上一層或同層，顯示路徑以供檢查
            continue

        target_csv = None
        files_in_folder = os.listdir(daily_folder)

        try:
            # 優先檢查並解壓 ZIP
            zips = [f for f in files_in_folder if f.endswith('.zip')]
            if zips:
                for z in zips:
                    with zipfile.ZipFile(os.path.join(daily_folder, z), 'r') as zf:
                        zf.extractall(temp_path)
                target_csv = os.path.join(temp_path, row['File'])

            # 若無 ZIP 則直接找 CSV
            if (not target_csv or not os.path.exists(target_csv)) and row['File'] in files_in_folder:
                target_csv = os.path.join(daily_folder, row['File'])

            if target_csv and os.path.exists(target_csv):
                with open(target_csv, 'rb') as f:
                    enc = chardet.detect(f.read(10000))['encoding']

                df = pd.read_csv(target_csv, encoding=enc)
                df.columns = df.columns.str.strip()

                # 欄位關鍵字識別
                c_type = next((c for c in df.columns if '買賣權' in c), None)
                c_exp = next((c for c in df.columns if '到期' in c), None)
                c_strike = next((c for c in df.columns if '履約' in c), None)
                c_price = next((c for c in df.columns if '成交價格' in c or '成交價' in c), None)
                c_vol = next((c for c in df.columns if '成交量' in c), None)

                mask = (df[c_type].str.contains('Call', na=False)) & (df[c_exp].astype(str).str.contains(target_contract))
                subset = df[mask].copy()

                if not subset.empty:
                    subset['IV'] = subset.apply(lambda x: calculate_iv(x[c_price], S0, x[c_strike], T, r_val), axis=1)
                    daily_avg = subset['IV'].mean()

                    res = pd.DataFrame({
                        'Date': [trade_date] * len(subset),
                        'Strike': subset[c_strike],
                        'Price': subset[c_price],
                        'Vol': subset[c_vol] if c_vol else 0,
                        'IV': subset['IV'],
                        '日平均IV': daily_avg
                    })
                    all_days_data.append(res)
                    print(f"✅ 成功處理：{trade_date}")

            if os.path.exists(temp_path):
                for f in os.listdir(temp_path): os.remove(os.path.join(temp_path, f))

        except Exception as e:
            print(f"❌ {trade_date} 處理失敗: {e}")

    # 4. 儲存結果與自動下載
    if all_days_data:
        final_df = pd.concat(all_days_data, ignore_index=True)
        out_name = 'Option_Analysis_Final.csv'
        # 儲存一份在 Colab
        final_df.to_csv(out_name, index=False, encoding='utf-8-sig')
        # 儲存一份在 Google Drive (索引檔同目錄)
        final_df.to_csv(os.path.join(base_path, out_name), index=False, encoding='utf-8-sig')
        print(f"\n🎉 任務完成！")
        print(f"1. 檔案已存回雲端：{os.path.join(base_path, out_name)}")
        print(f"2. 正在啟動瀏覽器下載...")
        files.download(out_name)
    else:
        print("\n💀 未能產生任何資料。請確認每日資料夾名稱是否與 Index 中的 File 欄位一致。")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🔍 正在為您搜尋檔案：Index_411336004_2022.csv ...
✅ 成功定位！索引檔位於：/content/drive/MyDrive/金融資料探勘/Index_411336004_2022.csv
📂 資料夾基準點已設為：/content/drive/MyDrive/金融資料探勘

💀 未能產生任何資料。請確認每日資料夾名稱是否與 Index 中的 File 欄位一致。


In [ ]:
import pandas as pd
import numpy as np
import os
import zipfile
import shutil
import chardet
from scipy.stats import norm
from google.colab import drive, files

# 1. 掛載 Google Drive
drive.mount('/content/drive')

# 2. 自動定位 Index 檔案
def find_path_auto(target_name):
    print(f"🔍 正在搜尋：{target_name} ...")
    for root, dirs, files_in_dir in os.walk('/content/drive/MyDrive/'):
        if target_name in files_in_dir:
            return os.path.join(root, target_name)
    return None

index_filename = 'Index_411336004_2022.csv'
full_index_path = find_path_auto(index_filename)

if not full_index_path:
    print(f"❌ 找不到索引檔。")
else:
    # --- 新增：自動尋找數據存放的真實根目錄 ---
    print("🔍 正在定位數據資料夾 (OptionsDaily_...) 的位置...")
    data_root_path = None
    # 嘗試在雲端硬碟中找第一個交易日資料夾
    test_folder = "OptionsDaily_2022_01_03"
    for root, dirs, _ in os.walk('/content/drive/MyDrive/'):
        if test_folder in dirs:
            data_root_path = root
            break

    if not data_root_path:
        # 如果找不到第一個，就用索引檔所在目錄當備案
        data_root_path = os.path.dirname(full_index_path)
        print(f"⚠️ 找不到測試資料夾，預設使用索引檔目錄。")

    print(f"✅ 成功定位數據根目錄：{data_root_path}")

    # --- 計算核心 ---
    def bs_call_price(S, K, T, r, sigma):
        if T <= 0 or sigma <= 0: return 0
        d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
        d2 = d1 - sigma * np.sqrt(T)
        return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

    def calculate_iv(market_price, S, K, T, r):
        low, high = 0.0001, 5.0
        for _ in range(50):
            mid = (low + high) / 2
            if bs_call_price(S, K, T, r, mid) < market_price: low = mid
            else: high = mid
        return mid

    # 3. 處理流程
    index_df = pd.read_csv(full_index_path)
    all_days_data = []
    temp_path = '/content/temp_extract/'
    if os.path.exists(temp_path): shutil.rmtree(temp_path)
    os.makedirs(temp_path)

    for i, row in index_df.iterrows():
        trade_date = row['Date']
        target_contract = str(row['Contract'])
        S0, r_val, T = row['S0'], row['Rf']/100, row['Maturity']/252

        # 資料夾名稱 (不帶 .csv)
        folder_name = row['File'].replace('.csv', '')
        daily_folder = os.path.join(data_root_path, folder_name)

        if not os.path.exists(daily_folder):
            # 輔助提示：印出找不到的路徑，方便糾錯
            if i < 3: print(f"⚠️ 找不到路徑：{daily_folder}")
            continue

        target_csv = None
        files_in_folder = os.listdir(daily_folder)

        try:
            # 優先解壓 ZIP
            zips = [f for f in files_in_folder if f.endswith('.zip')]
            if zips:
                for z in zips:
                    with zipfile.ZipFile(os.path.join(daily_folder, z), 'r') as zf:
                        zf.extractall(temp_path)
                target_csv = os.path.join(temp_path, row['File'])

            # 直接找 CSV
            if (not target_csv or not os.path.exists(target_csv)) and row['File'] in files_in_folder:
                target_csv = os.path.join(daily_folder, row['File'])

            if target_csv and os.path.exists(target_csv):
                with open(target_csv, 'rb') as f:
                    enc = chardet.detect(f.read(10000))['encoding']

                df = pd.read_csv(target_csv, encoding=enc)
                df.columns = df.columns.str.strip()

                # 自動抓取欄位
                c_type = next((c for c in df.columns if '買賣權' in c), None)
                c_exp = next((c for c in df.columns if '到期' in c), None)
                c_strike = next((c for c in df.columns if '履約' in c), None)
                c_price = next((c for c in df.columns if '成交價格' in c or '成交價' in c), None)
                c_vol = next((c for c in df.columns if '成交量' in c), None)

                # 篩選邏輯
                mask = (df[c_type].str.contains('Call', na=False)) & (df[c_exp].astype(str).str.contains(target_contract))
                subset = df[mask].copy()

                if not subset.empty:
                    subset['IV'] = subset.apply(lambda x: calculate_iv(x[c_price], S0, x[c_strike], T, r_val), axis=1)
                    daily_avg = subset['IV'].mean()

                    res = pd.DataFrame({
                        'Date': [trade_date] * len(subset),
                        'Strike': subset[c_strike],
                        'Price': subset[c_price],
                        'Vol': subset[c_vol] if col_vol else 0,
                        'IV': subset['IV'],
                        '日平均IV': daily_avg
                    })
                    all_days_data.append(res)
                    if i % 10 == 0: print(f"✅ 已處理到：{trade_date}")

            if os.path.exists(temp_path):
                for f in os.listdir(temp_path): os.remove(os.path.join(temp_path, f))

        except Exception as e:
            pass

    # 4. 存檔與下載
    if all_days_data:
        final_df = pd.concat(all_days_data, ignore_index=True)
        out_name = 'Option_Analysis_Final.csv'
        final_df.to_csv(out_name, index=False, encoding='utf-8-sig')
        print(f"\n🎉 處理完成！共 {len(final_df)} 筆資料。正在啟動下載...")
        files.download(out_name)
    else:
        print("\n💀 仍未找到資料。請確認資料夾是否位於：")
        print(f"{data_root_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🔍 正在搜尋：Index_411336004_2022.csv ...
🔍 正在定位數據資料夾 (OptionsDaily_...) 的位置...
✅ 成功定位數據根目錄：/content/drive/MyDrive/金融資料探勘/2022逐筆交易資料


/tmp/ipykernel_2307/1649225236.py:100: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(target_csv, encoding=enc)
/tmp/ipykernel_2307/1649225236.py:100: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(target_csv, encoding=enc)
/tmp/ipykernel_2307/1649225236.py:100: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(target_csv, encoding=enc)
/tmp/ipykernel_2307/1649225236.py:100: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(target_csv, encoding=enc)
/tmp/ipykernel_2307/1649225236.py:100: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(target_csv, encoding=enc)
/tmp/ipykernel_2307/1649225236.py:100: DtypeWarning: Columns (0) have mixed type

KeyboardInterrupt: 

In [ ]:
import pandas as pd
import numpy as np
import os
import zipfile
import shutil
import chardet
import warnings
from scipy.stats import norm
from google.colab import drive, files

# 忽略 DtypeWarning 讓介面乾淨
warnings.filterwarnings('ignore', category=pd.errors.DtypeWarning)

# 1. 掛載 Google Drive
drive.mount('/content/drive')

# 2. 自動尋找索引檔與數據目錄
def find_path_auto(target_name):
    print(f"🔍 正在搜尋：{target_name} ...")
    for root, dirs, files_in_dir in os.walk('/content/drive/MyDrive/'):
        if target_name in files_in_dir:
            return os.path.join(root, target_name)
    return None

index_filename = 'Index_411336004_2022.csv'
full_index_path = find_path_auto(index_filename)

if not full_index_path:
    print("❌ 找不到索引檔，請確認檔案已上傳。")
else:
    data_root_path = None
    # 偵測 OptionsDaily 資料夾位置
    for root, dirs, _ in os.walk('/content/drive/MyDrive/'):
        if any(d.startswith("OptionsDaily_2022") for d in dirs):
            data_root_path = root
            break

    print(f"✅ 索引檔位置：{full_index_path}")
    print(f"✅ 數據根目錄：{data_root_path}")

    # --- 金融計算函數 ---
    def bs_call_price(S, K, T, r, sigma):
        if T <= 0 or sigma <= 0: return max(0, S - K)
        d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
        d2 = d1 - sigma * np.sqrt(T)
        return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

    def calculate_iv(market_price, S, K, T, r):
        # 如果市價低於內含價值，IV 設為 0
        if market_price <= max(0, S - K * np.exp(-r * T)): return 0.0
        low, high = 0.0001, 3.0
        for _ in range(40): # 40次迭代足以達到高精度
            mid = (low + high) / 2
            if bs_call_price(S, K, T, r, mid) < market_price: low = mid
            else: high = mid
        return mid

    # 3. 開始處理
    index_df = pd.read_csv(full_index_path)
    all_days_data = []
    temp_path = '/content/temp_extract/'
    if os.path.exists(temp_path): shutil.rmtree(temp_path)
    os.makedirs(temp_path)

    print("\n--- 正在計算隱含波動率 (IV) ---")

    for i, row in index_df.iterrows():
        trade_date = row['Date']
        target_contract = str(row['Contract'])
        S0, r_val, T = row['S0'], row['Rf']/100, row['Maturity']/252

        folder_name = row['File'].replace('.csv', '')
        daily_folder = os.path.join(data_root_path, folder_name)

        if not os.path.exists(daily_folder): continue

        target_csv = None
        files_in_folder = os.listdir(daily_folder)

        try:
            # 處理壓縮或直接讀取
            zips = [f for f in files_in_folder if f.endswith('.zip')]
            if zips:
                with zipfile.ZipFile(os.path.join(daily_folder, zips[0]), 'r') as zf:
                    zf.extractall(temp_path)
                target_csv = os.path.join(temp_path, row['File'])
            elif row['File'] in files_in_folder:
                target_csv = os.path.join(daily_folder, row['File'])

            if target_csv and os.path.exists(target_csv):
                # 讀取資料並解決 DtypeWarning
                df = pd.read_csv(target_csv, encoding='cp950', low_memory=False)
                df.columns = df.columns.str.strip()

                # 自動識別中文欄位
                c_type = next((c for c in df.columns if '買賣權' in c), None)
                c_exp = next((c for c in df.columns if '到期' in c), None)
                c_strike = next((c for c in df.columns if '履約' in c), None)
                c_price = next((c for c in df.columns if '成交價格' in c or '成交價' in c), None)
                c_vol = next((c for c in df.columns if '成交量' in c), None)

                # 執行篩選
                mask = (df[c_type].str.contains('Call', na=False)) & (df[c_exp].astype(str).str.contains(target_contract))
                subset = df[mask].copy()

                if not subset.empty:
                    # 轉換為數值型態，避免計算錯誤
                    subset[c_price] = pd.to_numeric(subset[c_price], errors='coerce')
                    subset[c_strike] = pd.to_numeric(subset[c_strike], errors='coerce')
                    subset = subset.dropna(subset=[c_price, c_strike])

                    # 計算 IV
                    subset['IV'] = subset.apply(lambda x: calculate_iv(x[c_price], S0, x[c_strike], T, r_val), axis=1)
                    daily_avg = subset['IV'].mean()

                    # 組合結果
                    res = pd.DataFrame({
                        'Date': [trade_date] * len(subset),
                        'Strike': subset[c_strike],
                        'Price': subset[c_price],
                        'Vol': subset[c_vol] if c_vol else 0,
                        'IV': subset['IV'],
                        '日平均IV': [daily_avg] * len(subset)
                    })
                    all_days_data.append(res)
                    if i % 10 == 0: print(f"📈 進度：{trade_date} (OK)")

            # 清理
            if os.path.exists(temp_path):
                for f in os.listdir(temp_path): os.remove(os.path.join(temp_path, f))

        except Exception as e:
            print(f"⚠️ {trade_date} 略過：{str(e)[:50]}")

    # 4. 合併輸出
    if all_days_data:
        final_df = pd.concat(all_days_data, ignore_index=True)
        out_name = 'Option_IV_Results_2022.csv'
        final_df.to_csv(out_name, index=False, encoding='utf-8-sig')
        # 同步備份到 Google Drive (索引檔所在位置)
        final_df.to_csv(os.path.join(os.path.dirname(full_index_path), out_name), index=False, encoding='utf-8-sig')

        print(f"\n🎉 任務完成！共處理 {len(final_df)} 筆交易記錄。")
        print(f"💾 檔案已同步存至雲端硬碟：{out_name}")
        files.download(out_name)
    else:
        print("\n💀 未能產生任何資料，請檢查合約月份 (Contract) 格式是否匹配。")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🔍 正在搜尋：Index_411336004_2022.csv ...
✅ 索引檔位置：/content/drive/MyDrive/金融資料探勘/Index_411336004_2022.csv
✅ 數據根目錄：/content/drive/MyDrive/金融資料探勘/2022逐筆交易資料

--- 正在計算隱含波動率 (IV) ---

💀 未能產生任何資料，請檢查合約月份 (Contract) 格式是否匹配。


In [ ]:
import pandas as pd
import numpy as np
import os
import zipfile
import chardet
import warnings
from scipy.stats import norm
from google.colab import drive, files

warnings.filterwarnings('ignore')

drive.mount('/content/drive')

# 1. 定位檔案 (使用你剛才成功的路徑)
full_index_path = '/content/drive/MyDrive/金融資料探勘/Index_411336004_2022.csv'
data_root_path = '/content/drive/MyDrive/金融資料探勘/2022逐筆交易資料'

# 2. BS 與 IV 計算
def bs_call_price(S, K, T, r, sigma):
    if T <= 0 or sigma <= 0: return max(0, S - K)
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

def calculate_iv(market_price, S, K, T, r):
    if market_price <= max(0, S - K * np.exp(-r * T)): return 0.0
    low, high = 0.001, 4.0
    for _ in range(40):
        mid = (low + high) / 2
        if bs_call_price(S, K, T, r, mid) < market_price: low = mid
        else: high = mid
    return mid

# 3. 主處理邏輯
index_df = pd.read_csv(full_index_path)
all_days_data = []
temp_path = '/content/temp_extract/'
if os.path.exists(temp_path): import shutil; shutil.rmtree(temp_path)
os.makedirs(temp_path)

print(f"--- 開始處理，共 {len(index_df)} 個交易日 ---")

for i, row in index_df.iterrows():
    trade_date = row['Date']
    # 將合約轉為字串並去除可能的 .0 (例如 202201)
    target_contract = str(int(row['Contract']))
    S0, r_val, T = row['S0'], row['Rf']/100, row['Maturity']/252

    folder_name = row['File'].replace('.csv', '')
    daily_folder = os.path.join(data_root_path, folder_name)

    if not os.path.exists(daily_folder): continue

    target_csv = None
    files_in_folder = os.listdir(daily_folder)

    try:
        # 處理 Zip 或直接讀取
        zips = [f for f in files_in_folder if f.endswith('.zip')]
        if zips:
            with zipfile.ZipFile(os.path.join(daily_folder, zips[0]), 'r') as zf:
                zf.extractall(temp_path)
            target_csv = os.path.join(temp_path, row['File'])
        elif row['File'] in files_in_folder:
            target_csv = os.path.join(daily_folder, row['File'])

        if target_csv and os.path.exists(target_csv):
            # 使用 cp950 (Big5) 讀取台灣金融資料常見編碼
            df = pd.read_csv(target_csv, encoding='cp950', low_memory=False)
            df.columns = df.columns.str.strip()

            # --- 自動欄位辨識 ---
            c_type = next((c for c in df.columns if '買賣權' in c), None)
            c_exp = next((c for c in df.columns if '到期' in c), None)
            c_strike = next((c for c in df.columns if '履約' in c), None)
            c_price = next((c for c in df.columns if '成交價格' in c or '成交價' in c), None)
            c_vol = next((c for c in df.columns if '成交量' in c), None)

            # --- 強化過濾邏輯 ---
            # 1. 買賣權：包含 'C' 或 'Call' (不分大小寫)
            # 2. 合約：將 CSV 內容轉字串後，比對是否包含 target_contract (例如 '202201')
            df[c_exp] = df[c_exp].astype(str).str.strip()

            mask = (df[c_type].str.upper().str.contains('C', na=False)) & \
                   (df[c_exp].str.contains(target_contract, na=False))

            subset = df[mask].copy()

            if not subset.empty:
                subset[c_price] = pd.to_numeric(subset[c_price], errors='coerce')
                subset[c_strike] = pd.to_numeric(subset[c_strike], errors='coerce')
                subset = subset.dropna(subset=[c_price, c_strike])

                subset['IV'] = subset.apply(lambda x: calculate_iv(x[c_price], S0, x[c_strike], T, r_val), axis=1)
                daily_avg = subset['IV'].mean()

                res = pd.DataFrame({
                    'Date': [trade_date] * len(subset),
                    'Strike': subset[c_strike],
                    'Price': subset[c_price],
                    'Vol': subset[c_vol] if c_vol else 0,
                    'IV': subset['IV'],
                    '日平均IV': [daily_avg] * len(subset)
                })
                all_days_data.append(res)
                if i % 5 == 0: print(f"📈 成功進度：{trade_date}")

        # 清理暫存
        for f in os.listdir(temp_path): os.remove(os.path.join(temp_path, f))

    except Exception as e:
        continue

# 4. 輸出
if all_days_data:
    final_df = pd.concat(all_days_data, ignore_index=True)
    out_name = 'Option_IV_Final_Fixed.csv'
    final_df.to_csv(out_name, index=False, encoding='utf-8-sig')
    print(f"\n🎉 成功！處理了 {len(all_days_data)} 天，共 {len(final_df)} 筆資料。")
    files.download(out_name)
else:
    print("\n💀 仍然篩選不到資料。請檢查：")
    print(f"1. 索引檔合約 (如 202201) 是否出現在每日 CSV 的到期月份欄位中？")
    print(f"2. 每日 CSV 的買賣權欄位，Call 是寫成 'Call'、'C' 還是 '買權'？")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
--- 開始處理，共 246 個交易日 ---
📈 成功進度：2022/1/3


KeyboardInterrupt: 

In [ ]:
import pandas as pd
import numpy as np
import os
import zipfile
import chardet
import warnings
from scipy.stats import norm
from google.colab import drive, files

warnings.filterwarnings('ignore')
drive.mount('/content/drive')

# 1. 路徑設定 (沿用之前成功的路徑)
full_index_path = '/content/drive/MyDrive/金融資料探勘/Index_411336004_2022.csv'
data_root_path = '/content/drive/MyDrive/金融資料探勘/2022逐筆交易資料'

# 2. BS 與 IV 計算 (優化迭代次數)
def bs_call_price(S, K, T, r, sigma):
    if T <= 0 or sigma <= 0: return max(0, S - K)
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

def calculate_iv(market_price, S, K, T, r):
    if market_price <= max(0, S - K * np.exp(-r * T)): return 0.0
    low, high = 0.001, 3.0
    for _ in range(30): # 30次迭代已足夠精確且更快
        mid = (low + high) / 2
        if bs_call_price(S, K, T, r, mid) < market_price: low = mid
        else: high = mid
    return mid

# 3. 處理邏輯
index_df = pd.read_csv(full_index_path)
all_days_data = []
temp_path = '/content/temp_extract/'
if os.path.exists(temp_path): import shutil; shutil.rmtree(temp_path)
os.makedirs(temp_path)

print(f"--- 開始處理，將自動執行重複項優化以提升速度 ---")

for i, row in index_df.iterrows():
    trade_date = row['Date']
    target_contract = str(int(row['Contract']))
    S0, r_val, T = row['S0'], row['Rf']/100, row['Maturity']/252

    daily_folder = os.path.join(data_root_path, row['File'].replace('.csv', ''))
    if not os.path.exists(daily_folder): continue

    try:
        # 讀取檔案
        files_in_folder = os.listdir(daily_folder)
        target_csv = None
        zips = [f for f in files_in_folder if f.endswith('.zip')]
        if zips:
            with zipfile.ZipFile(os.path.join(daily_folder, zips[0]), 'r') as zf:
                zf.extractall(temp_path)
            target_csv = os.path.join(temp_path, row['File'])
        else:
            target_csv = os.path.join(daily_folder, row['File'])

        if target_csv and os.path.exists(target_csv):
            df = pd.read_csv(target_csv, encoding='cp950', low_memory=False)
            df.columns = df.columns.str.strip()

            # 欄位辨識
            c_type = next((c for c in df.columns if '買賣權' in c), None)
            c_exp = next((c for c in df.columns if '到期' in c), None)
            c_strike = next((c for c in df.columns if '履約' in c), None)
            c_price = next((c for c in df.columns if '成交價格' in c or '成交價' in c), None)
            c_vol = next((c for c in df.columns if '成交數量' in c), None)

            # --- 篩選條件：Call + 合約 + 交易量 > 30 ---
            df[c_vol] = pd.to_numeric(df[c_vol].astype(str).str.strip(), errors='coerce').fillna(0)

            mask = (df[c_type].str.upper().str.contains('C', na=False)) & \
                   (df[c_exp].astype(str).str.contains(target_contract)) & \
                   (df[c_vol] > 30)

            subset = df[mask].copy()

            if not subset.empty:
                # --- [優化關鍵]：合併重複的價格組合 ---
                # 只針對唯一的 (履約價, 成交價) 計算 IV
                subset[c_price] = pd.to_numeric(subset[c_price], errors='coerce')
                subset[c_strike] = pd.to_numeric(subset[c_strike], errors='coerce')
                subset = subset.dropna(subset=[c_price, c_strike])

                unique_iv_pairs = subset[[c_strike, c_price]].drop_duplicates()
                unique_iv_pairs['IV'] = unique_iv_pairs.apply(
                    lambda x: calculate_iv(x[c_price], S0, x[c_strike], T, r_val), axis=1
                )

                # 將計算好的 IV 合併回 subset
                subset = subset.merge(unique_iv_pairs, on=[c_strike, c_price], how='left')

                # 紀錄結果
                subset['Date'] = trade_date
                all_days_data.append(subset[['Date', c_strike, c_price, c_vol, 'IV']])
                print(f"📈 {trade_date} 完成 (篩選後剩餘 {len(subset)} 筆)")

        # 清理暫存
        if os.path.exists(temp_path):
            for f in os.listdir(temp_path): os.remove(os.path.join(temp_path, f))

    except Exception as e:
        print(f"⚠️ {trade_date} 錯誤: {e}")

# 4. 統計與輸出
if all_days_data:
    final_df = pd.concat(all_days_data, ignore_index=True)

    # --- 敘述性統計 ---
    stats = final_df.groupby('Date')['IV'].describe()
    print("\n--- 每日隱含波動率 (IV) 敘述性統計 ---")
    print(stats)

    # 儲存統計結果
    stats.to_csv('IV_Descriptive_Statistics.csv', encoding='utf-8-sig')
    final_df.to_csv('Option_IV_Volume30_Final.csv', index=False, encoding='utf-8-sig')

    print("\n🎉 處理完畢！正在下載統計報表與完整資料...")
    files.download('IV_Descriptive_Statistics.csv')
    files.download('Option_IV_Volume30_Final.csv')
else:
    print("\n💀 未能篩選到交易量 > 30 的資料。")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
--- 開始處理，將自動執行重複項優化以提升速度 ---
📈 2022/1/3 完成 (篩選後剩餘 1805 筆)
📈 2022/1/4 完成 (篩選後剩餘 2768 筆)
📈 2022/1/5 完成 (篩選後剩餘 2598 筆)
📈 2022/1/6 完成 (篩選後剩餘 1833 筆)
📈 2022/1/7 完成 (篩選後剩餘 2389 筆)
📈 2022/1/10 完成 (篩選後剩餘 2085 筆)
📈 2022/1/11 完成 (篩選後剩餘 2802 筆)
📈 2022/1/12 完成 (篩選後剩餘 2969 筆)
📈 2022/1/13 完成 (篩選後剩餘 1058 筆)
📈 2022/1/14 完成 (篩選後剩餘 2949 筆)
📈 2022/1/17 完成 (篩選後剩餘 2250 筆)
📈 2022/1/18 完成 (篩選後剩餘 71 筆)
📈 2022/1/19 完成 (篩選後剩餘 111 筆)
📈 2022/1/20 完成 (篩選後剩餘 244 筆)
📈 2022/1/21 完成 (篩選後剩餘 194 筆)
📈 2022/1/24 完成 (篩選後剩餘 156 筆)
📈 2022/1/25 完成 (篩選後剩餘 170 筆)
📈 2022/1/26 完成 (篩選後剩餘 373 筆)
📈 2022/2/7 完成 (篩選後剩餘 2418 筆)
📈 2022/2/8 完成 (篩選後剩餘 1296 筆)
📈 2022/2/9 完成 (篩選後剩餘 2107 筆)
📈 2022/2/10 完成 (篩選後剩餘 1219 筆)
📈 2022/2/11 完成 (篩選後剩餘 1270 筆)
📈 2022/2/14 完成 (篩選後剩餘 2560 筆)
📈 2022/2/15 完成 (篩選後剩餘 100 筆)
📈 2022/2/16 完成 (篩選後剩餘 103 筆)
📈 2022/2/17 完成 (篩選後剩餘 235 筆)
📈 2022/2/18 完成 (篩選後剩餘 79 筆)
📈 2022/2/21 完成 (篩選後剩餘 1

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import numpy as np
import os
import zipfile
import chardet
import warnings
from scipy.stats import norm
from google.colab import drive, files

# 忽略警告
warnings.filterwarnings('ignore')
drive.mount('/content/drive')

# 1. 路徑設定
full_index_path = '/content/drive/MyDrive/金融資料探勘/Index_411336004_2022.csv'
data_root_path = '/content/drive/MyDrive/金融資料探勘/2022逐筆交易資料'

# 2. BS 與 IV 計算 (優化版)
def bs_call_price(S, K, T, r, sigma):
    if T <= 0 or sigma <= 0: return max(0, S - K)
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

def calculate_iv(market_price, S, K, T, r):
    if market_price <= max(0, S - K * np.exp(-r * T)): return 0.0
    low, high = 0.001, 4.0
    for _ in range(35): # 35次迭代足以應對大部分金融計算
        mid = (low + high) / 2
        if bs_call_price(S, K, T, r, mid) < market_price: low = mid
        else: high = mid
    return mid

# 3. 處理流程
index_df = pd.read_csv(full_index_path)
all_days_data = []
temp_path = '/content/temp_extract/'
if os.path.exists(temp_path): import shutil; shutil.rmtree(temp_path)
os.makedirs(temp_path)

print(f"--- 啟動高效運算模式 (預計每交易日處理時間 < 10秒) ---")

for i, row in index_df.iterrows():
    trade_date = row['Date']
    target_contract = str(int(row['Contract']))
    S0, r_val, T = row['S0'], row['Rf']/100, row['Maturity']/252

    # 資料夾名稱不帶 .csv
    folder_name = row['File'].replace('.csv', '')
    daily_folder = os.path.join(data_root_path, folder_name)
    if not os.path.exists(daily_folder): continue

    try:
        # 解壓縮與讀取
        files_in_folder = os.listdir(daily_folder)
        target_csv = None
        zips = [f for f in files_in_folder if f.endswith('.zip')]
        if zips:
            with zipfile.ZipFile(os.path.join(daily_folder, zips[0]), 'r') as zf:
                zf.extractall(temp_path)
            target_csv = os.path.join(temp_path, row['File'])
        else:
            target_csv = os.path.join(daily_folder, row['File'])

        if target_csv and os.path.exists(target_csv):
            # 讀取資料
            df = pd.read_csv(target_csv, encoding='cp950', low_memory=False)
            df.columns = df.columns.str.strip()

            # 欄位自動識別
            c_type = next((c for c in df.columns if '買賣權' in c), None)
            c_exp = next((c for c in df.columns if '到期' in c), None)
            c_strike = next((c for c in df.columns if '履約' in c), None)
            c_price = next((c for c in df.columns if '成交價格' in c or '成交價' in c), None)
            c_vol = next((c for c in df.columns if '成交數量(B or S)' in c or '成交數量' in c), None)

            # A. 數據預處理：清洗成交量
            df[c_vol] = pd.to_numeric(df[c_vol].astype(str).str.strip(), errors='coerce').fillna(0)

            # B. 執行篩選 (Call + 正確月份 + 交易量 > 30)
            mask = (df[c_type].str.upper().str.contains('C', na=False)) & \
                   (df[c_exp].astype(str).str.contains(target_contract)) & \
                   (df[c_vol] > 30)
            subset = df[mask].copy()

            if not subset.empty:
                # C. 速度優化：只針對「唯一組合」計算 IV
                subset[c_price] = pd.to_numeric(subset[c_price], errors='coerce')
                subset[c_strike] = pd.to_numeric(subset[c_strike], errors='coerce')
                subset = subset.dropna(subset=[c_price, c_strike])

                # 提取唯一的履約價與價格對，進行 IV 計算
                unique_pairs = subset[[c_strike, c_price]].drop_duplicates()
                unique_pairs['IV'] = unique_pairs.apply(
                    lambda x: calculate_iv(x[c_price], S0, x[c_strike], T, r_val), axis=1
                )

                # 將計算結果對應回原始過濾後的表
                subset = subset.merge(unique_pairs, on=[c_strike, c_price], how='left')

                # D. 計算日平均 IV
                daily_avg = subset['IV'].mean()

                # E. 重新命名欄位並整理輸出格式
                res = subset.rename(columns={
                    c_strike: 'Strike',
                    c_price: 'Price',
                    c_vol: 'Vol'
                })
                res['Date'] = trade_date
                res['日平均IV'] = daily_avg

                # 選取要求的欄位
                final_cols = ['Date', 'Strike', 'Price', 'Vol', 'IV', '日平均IV']
                all_days_data.append(res[final_cols])

                print(f"✅ {trade_date} 完成：篩選出 {len(subset)} 筆，該日平均 IV: {daily_avg:.4f}")

        # 清理暫存
        if os.path.exists(temp_path):
            for f in os.listdir(temp_path): os.remove(os.path.join(temp_path, f))

    except Exception as e:
        print(f"⚠️ {trade_date} 處理出錯: {e}")

# 4. 合併輸出與統計
if all_days_data:
    final_output = pd.concat(all_days_data, ignore_index=True)

    # 產出敘述性統計檔案
    descriptive_stats = final_output.groupby('Date')['IV'].describe()

    # 下載檔案
    final_output.to_csv('Option_IV_Analysis_Final.csv', index=False, encoding='utf-8-sig')
    descriptive_stats.to_csv('IV_Descriptive_Stats.csv', encoding='utf-8-sig')

    print("\n🎉 分析全部完成！")
    print("正在下載：1. 完整資料(含Strike, Price, Vol...) 2. 每日IV敘述性統計")
    files.download('Option_IV_Analysis_Final.csv')
    files.download('IV_Descriptive_Stats.csv')
else:
    print("\n💀 失敗：未能篩選出任何符合條件 (Vol > 30) 的資料。")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
--- 啟動高效運算模式 (預計每交易日處理時間 < 10秒) ---
✅ 2022/1/3 完成：篩選出 1805 筆，該日平均 IV: 0.0484
✅ 2022/1/4 完成：篩選出 2768 筆，該日平均 IV: 0.0215
✅ 2022/1/5 完成：篩選出 2598 筆，該日平均 IV: 0.0284
✅ 2022/1/6 完成：篩選出 1833 筆，該日平均 IV: 0.0700
✅ 2022/1/7 完成：篩選出 2389 筆，該日平均 IV: 0.0918
✅ 2022/1/10 完成：篩選出 2085 筆，該日平均 IV: 0.0589
✅ 2022/1/11 完成：篩選出 2802 筆，該日平均 IV: 0.0396
✅ 2022/1/12 完成：篩選出 2969 筆，該日平均 IV: 0.0276
✅ 2022/1/13 完成：篩選出 1058 筆，該日平均 IV: 0.1021
✅ 2022/1/14 完成：篩選出 2949 筆，該日平均 IV: 0.1156
✅ 2022/1/17 完成：篩選出 2250 筆，該日平均 IV: 0.0890
✅ 2022/1/18 完成：篩選出 71 筆，該日平均 IV: 0.1331
✅ 2022/1/19 完成：篩選出 111 筆，該日平均 IV: 0.1169
✅ 2022/1/20 完成：篩選出 244 筆，該日平均 IV: 0.1172
✅ 2022/1/21 完成：篩選出 194 筆，該日平均 IV: 0.1328
✅ 2022/1/24 完成：篩選出 156 筆，該日平均 IV: 0.1095
✅ 2022/1/25 完成：篩選出 170 筆，該日平均 IV: 0.1393
✅ 2022/1/26 完成：篩選出 373 筆，該日平均 IV: 0.1175
✅ 2022/2/7 完成：篩選出 2418 筆，該日平均 IV: 0.0363
✅ 2022/2/8 完成：篩選出 1296 筆，該日平均 IV: 0.0657
✅ 2022/2/9

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>